In [28]:
import pandas as pd
import numpy as np

xml_path = "dades/Xml_20261119_124136_1.xml"

# Preprocessing

In this file we do the preprocessing of the data file `Xml_20261119_124136_1.xml`, which we obtained from a Government of Spain database. As this .xml file has the location, cause, time, and burned area stored differently, we are going to preprocess them independently, with the objective of merging them at the end.

## Location

In [29]:
location_columns = ["numeroparte", "idcomunidad", "idprovincia",
                    "idmunicipio", "latitud", "longitud"]

locations = pd.read_xml(xml_path, xpath="./Pif/pif_localizacion")[location_columns]
locations["codi_municipi"] = (
    locations["idprovincia"].astype("Int64").astype("string").str.zfill(2)
    + locations["idmunicipio"].astype("Int64").astype("string").str.zfill(3)
)
locations["codi_municipi"] = locations["codi_municipi"].mask(locations["idmunicipio"]==0)
locations = locations.drop(["idcomunidad", "idmunicipio"], axis=1)
locations["coords_imputades"] = (locations["latitud"].isna() & locations["codi_municipi"].notna())
locations.info()
locations.loc[locations["codi_municipi"].notna()].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15366 entries, 0 to 15365
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   numeroparte       15366 non-null  int64  
 1   idprovincia       15366 non-null  int64  
 2   latitud           14397 non-null  float64
 3   longitud          14397 non-null  float64
 4   codi_municipi     15366 non-null  string 
 5   coords_imputades  15366 non-null  bool   
dtypes: bool(1), float64(2), int64(2), string(1)
memory usage: 615.4 KB


,numeroparte,idprovincia,latitud,longitud,codi_municipi,coords_imputades
0,1998080001,8,41.411589,-4.328114,08085,False
1,1998080002,8,41.612949,-4.362242,08143,False
2,1998080003,8,41.732321,-4.397227,08170,False
3,1998080004,8,42.013889,-4.187195,08049,False
4,1998080005,8,41.733916,-4.414095,08189,False


In [30]:
provincies = ["08", "17", "25", "43"]
columnes = ["COD_INE", "COD_PROV",
            "NOMBRE_ACTUAL",
            "PROVINCIA",
            "LONGITUD_ETRS89_REGCAN95",
            "LATITUD_ETRS89_REGCAN95"]

path = "dades/MUNICIPIOS.csv"

municipis_info = pd.read_csv(
    path,
    encoding="cp1252",
    sep=";",
    dtype="string",
)

municipis_info = municipis_info.loc[municipis_info["COD_PROV"].isin(provincies), columnes]
municipis_info["codi_municipi"] = municipis_info["COD_INE"].str[:5]
municipis_info = municipis_info.drop(["COD_INE", "COD_PROV"], axis=1)
municipis_info.info()
municipis_info.head()

<class 'pandas.core.frame.DataFrame'>
Index: 947 entries, 862 to 6544
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   NOMBRE_ACTUAL             947 non-null    string
 1   PROVINCIA                 947 non-null    string
 2   LONGITUD_ETRS89_REGCAN95  947 non-null    string
 3   LATITUD_ETRS89_REGCAN95   947 non-null    string
 4   codi_municipi             947 non-null    string
dtypes: string(5)
memory usage: 44.4 KB


,NOMBRE_ACTUAL,PROVINCIA,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,codi_municipi
862,Abrera,Barcelona,"1,901568685","41,5163976",08001
863,Aguilar de Segarra,Barcelona,"1,631134116","41,73926837",08002
864,Alella,Barcelona,"2,295316458","41,49361157",08003
865,Alpens,Barcelona,"2,101098216","42,11928971",08004
866,L'Ametlla del Vallès,Barcelona,"2,261438098","41,66955386",08005


In [31]:
comarques = pd.read_csv(
    "dades/corrMpisComsAtps.csv",
    sep=";",
    skiprows=3,
    usecols=["Codi municipi", "Nom comarca", "Nom municipi"],
    dtype={"Codi municipi": "string"},
)#[["Codi municipi", "Nom comarca"]].copy()

comarques["codi_municipi"] = (
    comarques["Codi municipi"]
    .astype("string").str.strip()
    .str[:5]
    .str.zfill(5)
)
comarques = comarques[["codi_municipi", "Nom comarca"]]
comarques.info()
comarques.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947 entries, 0 to 946
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   codi_municipi  947 non-null    string
 1   Nom comarca    947 non-null    object
dtypes: object(1), string(1)
memory usage: 14.9+ KB


,codi_municipi,Nom comarca
0,25001,Pallars Jussà
1,08001,Baix Llobregat
2,25002,Noguera
3,25003,Urgell
4,08002,Bages


In [32]:
merged1 = locations.merge(
    municipis_info, 
    how="left", 
    on="codi_municipi")
merged1["latitud"] = merged1["latitud"].fillna(merged1["LATITUD_ETRS89_REGCAN95"])
merged1["longitud"] = merged1["longitud"].fillna(merged1["LONGITUD_ETRS89_REGCAN95"])
merged1 = merged1.drop(["LONGITUD_ETRS89_REGCAN95", "LATITUD_ETRS89_REGCAN95"], axis=1)
df_loc = merged1.merge(
    comarques,
    how="left",
    on="codi_municipi"
)
df_loc = df_loc.rename(columns={
    "NOMBRE_ACTUAL": "municipi",
    "PROVINCIA": "provincia",
    "Nom comarca": "comarca" 
})
df_loc = df_loc.iloc[:, [0,3,5,7,6,1,2,4]]
df_loc.loc[merged1["codi_municipi"].notna()].head()


,numeroparte,longitud,coords_imputades,provincia,municipi,idprovincia,latitud,codi_municipi
0,1998080001,-4.328114,False,Barcelona,Font-rubí,8,41.411589,08085
1,1998080002,-4.362242,False,Barcelona,Òdena,8,41.612949,08143
2,1998080003,-4.397227,False,Barcelona,Els Prats de Rei,8,41.732321,08170
3,1998080004,-4.187195,False,Barcelona,Casserres,8,42.013889,08049
4,1998080005,-4.414095,False,Barcelona,Sant Pere Sallavinera,8,41.733916,08189


In [33]:
df_loc.loc[(df_loc["municipi"] == "Tordera") & (df_loc["coords_imputades"] == False), ["municipi", "latitud", "longitud"]]

,municipi,latitud,longitud
208,Tordera,41.689955,2.688771
213,Tordera,41.704561,2.771634
272,Tordera,41.657408,2.646891
1873,Tordera,41.694518,2.711582
1886,Tordera,41.723349,2.71506
...,...,...,...
11900,Tordera,41.682202,2.713944
11930,Tordera,41.712664,2.721118
11956,Tordera,41.657089,2.642665
13402,Tordera,41.718887,2.736154


## Causes

In [34]:
causes = pd.read_xml(xml_path, xpath="./Pif/pif_causa")
causes.info()
causes.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15366 entries, 0 to 15365
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   numeroparte              15366 non-null  int64  
 1   idcausa                  15366 non-null  int64  
 2   idcertidumbrecausa       15366 non-null  int64  
 3   idcausante               15366 non-null  int64  
 4   idclasedia               15366 non-null  int64  
 5   idmotivacion             3765 non-null   float64
 6   diastormenta             3688 non-null   float64
 7   idgradoresponsabilidad   3583 non-null   float64
 8   idinvestigacioncausa     3583 non-null   float64
 9   idautorizacionactividad  3583 non-null   float64
 10  motivacionotros          447 non-null    object 
 11  causaotros               35 non-null     object 
dtypes: float64(5), int64(5), object(2)
memory usage: 1.4+ MB


,numeroparte,idcausa,idcertidumbrecausa,idcausante,idclasedia,idmotivacion,diastormenta,idgradoresponsabilidad,idinvestigacioncausa,idautorizacionactividad,motivacionotros,causaotros
0,1998080001,230,1,1,4,NaN,NaN,NaN,NaN,NaN,None,None
1,1998080002,210,1,1,4,NaN,NaN,NaN,NaN,NaN,None,None
2,1998080003,400,2,2,4,401.0,NaN,NaN,NaN,NaN,None,None
3,1998080004,500,1,2,2,NaN,NaN,NaN,NaN,NaN,None,None
4,1998080005,400,2,2,2,401.0,NaN,NaN,NaN,NaN,None,None


In [35]:
columnes_causes = ["numeroparte", "idcausa", "causa"]

causes_egif_ca = pd.read_csv("dades/causes_egif_ca.csv")
df_causes = causes.merge(
    causes_egif_ca,
    on="idcausa",
    how="left",
    validate="many_to_one",
)
df_causes["causa"] = df_causes["causa"].mask(df_causes["causa"] == "Desconeguda")
df_causes = df_causes[columnes_causes]
df_causes.head()


,numeroparte,idcausa,causa
0,1998080001,230,Altres cremes en treballs forestals (conegudes)
1,1998080002,210,Crema agrícola
2,1998080003,400,Intencionat
3,1998080004,500,NaN
4,1998080005,400,Intencionat


## Temps

In [36]:
temps_columnes = ["numeroparte", "deteccion"]
df_temps = pd.read_xml(xml_path, xpath="./Pif/pif_tiempos")[temps_columnes]
df_temps.head()

,numeroparte,deteccion
0,1998080001,1998-01-20T17:12:00
1,1998080002,1998-01-21T13:00:00
2,1998080003,1998-02-11T16:50:00
3,1998080004,1998-02-14T16:45:00
4,1998080005,1998-02-14T19:15:00


In [37]:
df_temps["deteccion"] = df_temps["deteccion"].astype("string").str.strip().str[:10]
df_temps["deteccion"] = pd.to_datetime(df_temps["deteccion"], errors="coerce", format="%Y-%m-%d")
df_temps = df_temps.rename(columns={
    "deteccion": "data"
})
df_temps.head()

,numeroparte,data
0,1998080001,1998-01-20
1,1998080002,1998-01-21
2,1998080003,1998-02-11
3,1998080004,1998-02-14
4,1998080005,1998-02-14


## Hectares

Total recorded burned area includes wooded forest, non-wooded forest, agricultural, and other non-forest land, in hectares. Missing components are skipped, and totals with no recorded area remain missing.

In [38]:
hectares_columnes = [
    "superficiearboladatotal",
    "superficienoarboladatotal",
    "superficienoarboladaagricola",
    "superficienoarboladaotras",
]

hectares = pd.read_xml(xml_path, xpath="./Pif/pif_perdidas")
hectares = hectares.reindex(columns=["numeroparte"] + hectares_columnes)
hectares["total_hectares"] = hectares[hectares_columnes].sum(axis=1, min_count=1)
df_hectares = hectares[["numeroparte", "total_hectares"]]
df_hectares.info()
df_hectares.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15366 entries, 0 to 15365
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   numeroparte     15366 non-null  int64  
 1   total_hectares  15366 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 240.2 KB


,numeroparte,total_hectares
0,1998080001,0.09
1,1998080002,0.30
2,1998080003,0.01
3,1998080004,2.00
4,1998080005,0.01


## Merge and export csv file

In [39]:
merged1 = df_loc.merge(
    df_causes,
    how="inner",
    on="numeroparte")
df = merged1.merge(
    df_temps,
    how="inner",
    on="numeroparte"
)

df = df.merge(
    df_hectares,
    how="left",
    on="numeroparte",
    validate="one_to_one",
)

df.to_csv("Clean data/CleanGovEsp.csv")
df.head()

,numeroparte,longitud,coords_imputades,provincia,municipi,idprovincia,latitud,codi_municipi,idcausa,causa,data,total_hectares
0,1998080001,-4.328114,False,Barcelona,Font-rubí,8,41.411589,08085,230,Altres cremes en treballs forestals (conegudes),1998-01-20,0.09
1,1998080002,-4.362242,False,Barcelona,Òdena,8,41.612949,08143,210,Crema agrícola,1998-01-21,0.30
2,1998080003,-4.397227,False,Barcelona,Els Prats de Rei,8,41.732321,08170,400,Intencionat,1998-02-11,0.01
3,1998080004,-4.187195,False,Barcelona,Casserres,8,42.013889,08049,500,NaN,1998-02-14,2.00
4,1998080005,-4.414095,False,Barcelona,Sant Pere Sallavinera,8,41.733916,08189,400,Intencionat,1998-02-14,0.01
